# naver에서 크롤링

In [ ]:
!pip install selenium requests

In [9]:
!pip install undetected-chromedriver

     ---------------------------------------- 0.0/65.4 kB ? eta -:--:--
     ---------------------------------------- 65.4/65.4 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/178.7 kB ? eta -:--:--
   -------------------------------------- - 174.1/178.7 kB ? eta -:--:--
   ---------------------------------------- 178.7/178.7 kB 3.6 MB/s eta 0:00:00
  Created wheel for undetected_chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47130 sha256=57d613132fb6e513e6d8e2534bd804108def4b63b022188734a9e7a76ef1d8e9
  Stored in directory: c:\users\1004t\appdata\local\pip\cache\wheels\c4\f1\aa\9de6cf276210554d91e9c0526864563e850a428c5e76da4914
Successfully built undetected_chromedriver


In [27]:
import os
import shutil
import time
import base64
import requests
from urllib.parse import quote

# Selenium 관련 라이브러리
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys

# 크롤링 제한 우회
import undetected_chromedriver as uc
import time

# 사진 크기 필터링
from io import BytesIO
from PIL import Image

start = time.time()

# ==========================================
# 사용자 설정 및 경로
# ==========================================
BASE_PATH = 'C:\\Users\\1004t\\Documents\\카카오톡 받은 파일'
QUERY_FILE_PATH = os.path.join(BASE_PATH, '한국 연예인1.txt')

LIMITS = {
    "정면": 100,      # 검색어 + 정면 20장
    "측면": 50     # 검색어 + 옆모습 20장
}

# 💡 최소 이미지 크기 설정 (이 값보다 작으면 필터링)
MIN_WIDTH = 200
MIN_HEIGHT = 200


# 2. 텍스트 파일에서 쿼리 읽어오기
if not os.path.exists(QUERY_FILE_PATH):
    os.makedirs(BASE_PATH, exist_ok=True)
    with open(QUERY_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write("수지\n아이유\n")
    print(f"[{QUERY_FILE_PATH}] 예시 파일이 생성되었습니다.")

with open(QUERY_FILE_PATH, 'r', encoding='utf-8') as f:
    queries = [line.strip() for line in f if line.strip()]

print(f"불러온 쿼리 목록: {queries}\n")

# 실습용 타겟 지정 (전체 실행 시 주석 처리)
# queries = ["한소희"]

# ==========================================
# 3. Selenium 웹드라이버 설정 (Colab 전용 Headless)
# ==========================================
chrome_options = Options()
# chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--window-size=1920,1080')
# Colab 내장 크롬 드라이버 경로
driver = uc.Chrome(options=chrome_options, version_main=147)

# ==========================================
# 4. 크롤링 및 다운로드 실행
# ==========================================
for query in queries:
    query_start = time.time()
    query_dir = os.path.join(BASE_PATH, "데이터셋", "한국 연예인", query)
    print(f"\n========================================\n대상: {query}\n저장 경로: {query_dir}\n========================================")
    
    # 안전하게 기존 폴더 삭제 후 재생성
    if os.path.exists(query_dir):
        shutil.rmtree(query_dir)
    os.makedirs(query_dir, exist_ok=True)
    
    search_types = ["정면", "측면"]
    
    for s_type in search_types:
#         search_query = f"{query}" if s_type == "얼굴" else f"{query} {s_type}"
        search_query = f"{query} {s_type} 고화질"
        limit_count = LIMITS[s_type]
        
        print(f"\n>>> '{search_query}' 검색 및 스크롤 시작... (목표: {limit_count}장)")
        
        # 구글 이미지 검색 URL 접속
        url = f"https://search.naver.com/search.naver?sm=tab_hty.top&where=image&ssc=tab.image.all&query={quote(search_query)}"
        driver.get(url)
        
        try:
            # 1단계: 브라우저 자체가 "나 로딩 다 했어(complete)"라고 신호를 보낼 때까지 최대 15초 대기
            WebDriverWait(driver, 15).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
            
            # 2단계: 우리가 크롤링할 구글 이미지 썸네일('.YQ4gaf')이 화면에 나타날 때까지 최대 15초 대기
            # 만약 1.5초 만에 이미지가 나타나면 15초를 채우지 않고 즉시 다음 줄로 넘어갑니다! (시간 엄청 절약됨)
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "._fe_image_tab_content_thumbnail_image"))
            )
            print("✔️ 초기 이미지 로딩 완료!")
            
        except Exception as e:
            print(f"⚠️ 페이지 로딩 타임아웃! 인터넷 연결을 확인하세요.")
            continue # 로딩 실패 시 에러 뿜지 않고 다음 검색어(옆모습 등)로 부드럽게 넘어감

            
        # 💡 [개선] 스크롤 흔들기 (Jiggle) 기법 적용
        last_height = driver.execute_script("return document.body.scrollHeight")
        for i in range(4): 
            # 1. 맨 밑으로 내리기
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            # 2. 네이버 지연 로딩 센서를 자극하기 위해 살짝 위로 올렸다가 다시 내리기
            driver.execute_script("window.scrollBy(0, -500);")
            time.sleep(1)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(3)
            
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
            print(f"  🔄 스크롤 진행 중 ({i+1}/4)")

        # 구글 이미지의 썸네일 클래스인 'rg_i'를 찾아 추출
        image_elements = driver.find_elements(By.CSS_SELECTOR, "._fe_image_tab_content_thumbnail_image")
        print(len(image_elements))
        
        saved_count = 1
        for img in image_elements:
            if saved_count > limit_count:
                break
                
            try:
                # 이미지 소스 URL 가져오기 (src가 없으면 data-src 확인)
                src = img.get_attribute('src')
                if not src:
                    src = img.get_attribute('data-src')
                    
                if not src:
                    continue

                # 우리가 원하는 파일명으로 즉시 저장
                # 확장자는 기본적으로 jpg로 통일하여 안전하게 저장합니다.
                file_name = f"{query}_{s_type}_{saved_count}.jpg"
                file_path = os.path.join(query_dir, file_name)

                img_data = None

                # 1. Base64 이미지 데이터 추출
                if src.startswith('data:image'):
                    base64_data = src.split(',')[1]
                    img_data = base64.b64decode(base64_data)
                
                # 2. 일반 HTTP URL 이미지 데이터 추출
                elif src.startswith('http'):
                    response = requests.get(src, timeout=10)
                    if response.status_code == 200:
                        img_data = response.content

                # 💡 [핵심] 이미지 크기 검사 로직 추가
                if img_data:
                    # 메모리 내에서 바이너리 데이터를 이미지 객체로 전환 (디스크 저장 전 검사)
                    with Image.open(BytesIO(img_data)) as pillow_img:
                        width, height = pillow_img.size
                        
                        # 둘 중 하나라도 설정해둔 해상도 이상이면 통과
                        if width < MIN_WIDTH or height < MIN_HEIGHT:
                            # 디버깅 편의를 위해 로그 출력
                            # print(f"⏩ 너무 작은 이미지 스킵 ({width}x{height})")
                            continue
                    
                    # 크기 통과 시 최종 저장
                    with open(file_path, 'wb') as f:
                        f.write(img_data)
                        saved_count += 1
                
            except Exception as e:
                # 다운로드 중 에러 발생 시 무시하고 다음 이미지로 진행
                print(e)
                continue

        print(f"     └─ {s_type} 사진 {saved_count - 1}장 저장 완료!")
    print(f"{query} : {(time.time()-query_start)} 초")

print("\n크롤링이 모두 완료되었습니다! 브라우저를 종료합니다.")
print(f"{(time.time() - start) / 60} 분")
driver.quit()

불러온 쿼리 목록: ['한소희', '류진', '비투비 이민혁', '투바투 수빈', '남궁민', '나재민', '마동석', '이영애', '송강호', '수지', '현아', '카리나', '이효리', '뷔', '박보검', '김고은', '장원영', '박지훈', '설윤', '유해진']


대상: 한소희
저장 경로: C:\Users\1004t\Documents\카카오톡 받은 파일\데이터셋\한국 연예인\한소희

>>> '한소희 정면 고화질' 검색 및 스크롤 시작... (목표: 100장)
✔️ 초기 이미지 로딩 완료!
  🔄 스크롤 진행 중 (1/4)
  🔄 스크롤 진행 중 (2/4)
  🔄 스크롤 진행 중 (3/4)
500
     └─ 정면 사진 100장 저장 완료!

>>> '한소희 측면 고화질' 검색 및 스크롤 시작... (목표: 50장)
✔️ 초기 이미지 로딩 완료!
  🔄 스크롤 진행 중 (1/4)
  🔄 스크롤 진행 중 (2/4)
  🔄 스크롤 진행 중 (3/4)
500
     └─ 측면 사진 50장 저장 완료!
한소희 : 71.9382095336914 초

대상: 류진
저장 경로: C:\Users\1004t\Documents\카카오톡 받은 파일\데이터셋\한국 연예인\류진

>>> '류진 정면 고화질' 검색 및 스크롤 시작... (목표: 100장)
✔️ 초기 이미지 로딩 완료!
  🔄 스크롤 진행 중 (1/4)
  🔄 스크롤 진행 중 (2/4)
  🔄 스크롤 진행 중 (3/4)
500
     └─ 정면 사진 100장 저장 완료!

>>> '류진 측면 고화질' 검색 및 스크롤 시작... (목표: 50장)
✔️ 초기 이미지 로딩 완료!
  🔄 스크롤 진행 중 (1/4)
  🔄 스크롤 진행 중 (2/4)
  🔄 스크롤 진행 중 (3/4)
500
     └─ 측면 사진 50장 저장 완료!
류진 : 61.21248769760132 초

대상: 비투비 이민혁
저장 경로: C:\Users\1004t\Documents\카카오톡 받은 파일\데이터셋\한국 연예인\비투비 이민혁


# 레거시

In [29]:
from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.support import expected_conditions as EC
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time



In [8]:
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('lang=ko_KR')
chrome_options.add_argument('user-agent=' + user_agent)

In [11]:
# 크롬 브라우저 열기
driver = webdriver.Chrome(options=chrome_options)
url = 'https://www.google.co.kr/imghp?hl=ko&ogbl'
driver.get(url)

In [12]:
input_element = driver.find_element(By.CLASS_NAME,"gLFyf")
input_element.send_keys("장원영" + Keys.ENTER)

In [18]:
driver.get(url)
input_element = driver.find_element(By.CLASS_NAME,"gLFyf")
input_element.send_keys("안유진" + Keys.ENTER)

In [27]:
import undetected_chromedriver as uc
options = uc.ChromeOptions()
driver = uc.Chrome(options=options, version_main=145)
driver.get("https://www.google.co.kr/imghp?hl=ko&ogbl")

In [21]:
!pip install undetected-chromedriver

     ---------------------------------------- 0.0/65.4 kB ? eta -:--:--
     ---------------------------------------- 65.4/65.4 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47130 sha256=eb9de9f6b16e7e97a96b759fefdc9bc2d9426c1eb211f500518375d0d5377221
  Stored in directory: c:\users\1004t\appdata\local\pip\cache\wheels\5c\b9\03\4b6e38f019d6170e8c25df2e1e362d7bdf9ff4012df2dc85c0
Successfully built undetected-chromedriver


In [29]:
input_element = driver.find_element(By.CLASS_NAME,"gLFyf")
input_element.send_keys("장원영" + Keys.ENTER)

In [30]:
input_element = driver.find_element(By.CLASS_NAME,"gLFyf")
input_element.send_keys("안유진" + Keys.ENTER)

ElementNotInteractableException: Message: element not interactable
  (Session info: chrome=145.0.7632.160)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x10c7dd3
	0x10c7e14
	0xed1bee
	0xf16e0f
	0xf1547b
	0xf3ea3c
	0xf118c4
	0xf3ec04
	0xf5b621
	0xf3e7d6
	0xf10049
	0xf10e04
	0x1326924
	0x1321bf7
	0x133f5a0
	0x10e0f58
	0x10e891d
	0x10d0648
	0x10d0812
	0x10ba21a
	0x76185d49
	0x77b7d83b
	0x77b7d7c1


In [31]:
members = ["장원영", "안유진", "카리나", "윈터"]

In [5]:
import undetected_chromedriver as uc
import time
import random
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = uc.ChromeOptions()
driver = uc.Chrome(options=options, version_main=145)
url = "https://www.google.com"

members = ["장원영", "안유진", "카리나", "윈터"]

try:
    # 처음 한 번만 구글 메인 접속
    driver.get(url)
    wait = WebDriverWait(driver, 15)

    for name in members:
        # 1. 검색창 찾기 (페이지가 바뀔 때마다 새로 찾아야 에러가 안 남)
        search_box = wait.until(EC.element_to_be_clickable((By.NAME, "q")))
        
        # 2. 기존 입력값 지우기 (가장 안전한 전체 선택 후 삭제 방식)
        search_box.send_keys(Keys.CONTROL + 'a')
        search_box.send_keys(Keys.BACKSPACE)
        
        # 3. 새 검색어 입력
        for char in name:
            search_box.send_keys(char)
            time.sleep(random.uniform(0.1, 0.2))
        
        search_box.send_keys(Keys.ENTER)
        print(f"'{name}' 검색 완료")
        
        # 4. 결과 로딩 및 봇 감지 회피를 위한 대기
        time.sleep(random.uniform(2, 4))

finally:
    driver.quit()

'장원영' 검색 완료
'안유진' 검색 완료
'카리나' 검색 완료
'윈터' 검색 완료


In [ ]:
def selenium_scroll_option():
  SCROLL_PAUSE_SEC = 3

  # 스크롤 높이 가져옴
  last_height = driver.execute_script("return document.body.scrollHeight")

  while True:
    # 끝까지 스크롤 다운
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    # 10초 대기
    time.sleep(SCROLL_PAUSE_SEC)

    # 스크롤 다운 후 스크롤 높이 다시 가져옴
    new_height = driver.execute_script("return document.body.scrollHeight")

    if new_height == last_height:
        break
    last_height = new_height

In [14]:
import os
import shutil
import time
import base64
import requests
from urllib.parse import quote

# Selenium 관련 라이브러리
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 크롤링 제한 우회
import undetected_chromedriver as uc
import time

# 사진 크기 필터링
from io import BytesIO
from PIL import Image

start = time.time()

# ==========================================
# 사용자 설정 및 경로
# ==========================================
BASE_PATH = 'C:\\Users\\1004t\\Desktop\\중앙대'
QUERY_FILE_PATH = os.path.join(BASE_PATH, '한국 연예인1.txt')

LIMITS = {
    "정면": 40,      # 검색어 + 정면 20장
    "옆모습": 30     # 검색어 + 옆모습 20장
}

# 💡 최소 이미지 크기 설정 (이 값보다 작으면 필터링)
MIN_WIDTH = 200
MIN_HEIGHT = 200


# 2. 텍스트 파일에서 쿼리 읽어오기
if not os.path.exists(QUERY_FILE_PATH):
    os.makedirs(BASE_PATH, exist_ok=True)
    with open(QUERY_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write("수지\n아이유\n")
    print(f"[{QUERY_FILE_PATH}] 예시 파일이 생성되었습니다.")

with open(QUERY_FILE_PATH, 'r', encoding='utf-8') as f:
    queries = [line.strip() for line in f if line.strip()]

print(f"불러온 쿼리 목록: {queries}\n")

# 실습용 타겟 지정 (전체 실행 시 주석 처리)
queries = ["손석구"]

# ==========================================
# 3. Selenium 웹드라이버 설정 (Colab 전용 Headless)
# ==========================================
chrome_options = Options()
# chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
# Colab 내장 크롬 드라이버 경로
driver = uc.Chrome(options=chrome_options, version_main=145)

# ==========================================
# 4. 크롤링 및 다운로드 실행
# ==========================================
for query in queries:
    query_start = time.time()
    query_dir = os.path.join(BASE_PATH, "데이터셋", "한국 연예인1", query)
    print(f"\n========================================\n대상: {query}\n저장 경로: {query_dir}\n========================================")
    
    # 안전하게 기존 폴더 삭제 후 재생성
    if os.path.exists(query_dir):
        shutil.rmtree(query_dir)
    os.makedirs(query_dir, exist_ok=True)
    
    search_types = ["정면", "옆모습"]
    
    for s_type in search_types:
#         search_query = f"{query}" if s_type == "얼굴" else f"{query} {s_type}"
        search_query = f"{query} {s_type} 고화질"
        limit_count = LIMITS[s_type]
        
        print(f"\n>>> '{search_query}' 검색 및 스크롤 시작... (목표: {limit_count}장)")
        
        # 구글 이미지 검색 URL 접속
        url = f"https://www.google.com/search?q={quote(search_query)}&tbm=isch"
        driver.get(url)
        
        try:
            # 1단계: 브라우저 자체가 "나 로딩 다 했어(complete)"라고 신호를 보낼 때까지 최대 15초 대기
            WebDriverWait(driver, 15).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
            
            # 2단계: 우리가 크롤링할 구글 이미지 썸네일('.YQ4gaf')이 화면에 나타날 때까지 최대 15초 대기
            # 만약 1.5초 만에 이미지가 나타나면 15초를 채우지 않고 즉시 다음 줄로 넘어갑니다! (시간 엄청 절약됨)
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".YQ4gaf"))
            )
            print("✔️ 초기 이미지 로딩 완료!")
            
        except Exception as e:
            print(f"⚠️ 페이지 로딩 타임아웃! 인터넷 연결을 확인하세요.")
            continue # 로딩 실패 시 에러 뿜지 않고 다음 검색어(옆모습 등)로 부드럽게 넘어감
            
        # 동적 페이지 스크롤 (이미지를 충분히 불러오기 위해)
        last_height = driver.execute_script("return document.body.scrollHeight")
        for _ in range(5): # 목표 개수에 따라 스크롤 횟수 조절 가능
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(7)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height

        # 구글 이미지의 썸네일 클래스인 'rg_i'를 찾아 추출
        image_elements = driver.find_elements(By.CSS_SELECTOR, ".YQ4gaf")
        
        saved_count = 1
        for img in image_elements:
            if saved_count > limit_count:
                break
                
            try:
                # 이미지 소스 URL 가져오기 (src가 없으면 data-src 확인)
                src = img.get_attribute('src')
                if not src:
                    src = img.get_attribute('data-src')
                    
                if not src:
                    continue

                # 우리가 원하는 파일명으로 즉시 저장
                # 확장자는 기본적으로 jpg로 통일하여 안전하게 저장합니다.
                file_name = f"{query}_{s_type}_{saved_count}.jpg"
                file_path = os.path.join(query_dir, file_name)

                img_data = None

                # 1. Base64 이미지 데이터 추출
                if src.startswith('data:image'):
                    base64_data = src.split(',')[1]
                    img_data = base64.b64decode(base64_data)
                
                # 2. 일반 HTTP URL 이미지 데이터 추출
                elif src.startswith('http'):
                    response = requests.get(src, timeout=10)
                    if response.status_code == 200:
                        img_data = response.content

                # 💡 [핵심] 이미지 크기 검사 로직 추가
                if img_data:
                    # 메모리 내에서 바이너리 데이터를 이미지 객체로 전환 (디스크 저장 전 검사)
                    with Image.open(BytesIO(img_data)) as pillow_img:
                        width, height = pillow_img.size
                        
                        # 둘 중 하나라도 설정해둔 해상도 이상이면 통과
                        if width < MIN_WIDTH and height < MIN_HEIGHT:
                            # 디버깅 편의를 위해 로그 출력
                            # print(f"⏩ 너무 작은 이미지 스킵 ({width}x{height})")
                            continue
                    
                    # 크기 통과 시 최종 저장
                    with open(file_path, 'wb') as f:
                        f.write(img_data)
                        saved_count += 1
                
            except Exception as e:
                # 다운로드 중 에러 발생 시 무시하고 다음 이미지로 진행
                print(e)
                continue

        print(f"     └─ {s_type} 사진 {saved_count - 1}장 저장 완료!")
    print(f"{query} : {(time.time()-query_start)} 초")

print("\n크롤링이 모두 완료되었습니다! 브라우저를 종료합니다.")
print(f"{(time.time() - start) / 60} 분")
driver.quit()

불러온 쿼리 목록: ['한소희', '류진', '비투비 이민혁', '투바투 수빈', '남궁민', '나재민', '마동석', '이영애', '송강호', '수지', '현아', '카리나', '이효리', '정지훈', '박보검', '김고은', '장원영', '박지훈', '설윤', '유해진']


대상: 손석구
저장 경로: C:\Users\1004t\Desktop\중앙대\데이터셋\한국 연예인1\손석구

>>> '손석구 정면 고화질' 검색 및 스크롤 시작... (목표: 40장)
✔️ 초기 이미지 로딩 완료!
     └─ 정면 사진 40장 저장 완료!

>>> '손석구 옆모습 고화질' 검색 및 스크롤 시작... (목표: 30장)
✔️ 초기 이미지 로딩 완료!
     └─ 옆모습 사진 30장 저장 완료!
손석구 : 24.39849090576172 초

크롤링이 모두 완료되었습니다! 브라우저를 종료합니다.
0.4227993726730347 분
